In [1]:
!pip install -q pdfplumber
import pdfplumber
import pandas as pd
import re
import os
import sys
import json
import urllib.request
from datetime import datetime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 931.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 91.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [2]:
# Modelo de datos

class ParametrosHidro:
    """Clase de datos para almacenar los parámetros extraídos y calculados."""
    def __init__(self):
        self.datos_base = {}
        self.datos_actualizados = {}
        self.metadatos = {}

In [3]:

# Extraemos la información de los pdfs con "pdfplumber"

class IDAEPDFExtractor:
    """Se encarga de abrir el PDF y buscar los patrones de texto."""

    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self._verificar_archivo()

    def _verificar_archivo(self):
        if not os.path.exists(self.pdf_path):
            alt_path = os.path.join(os.path.dirname(os.getcwd()), self.pdf_path)
            if not os.path.exists(alt_path):
                raise FileNotFoundError(f"No se encuentro el PDF en la ruta: {self.pdf_path}")
            self.pdf_path = alt_path

    def extraer_parametros(self) -> dict:
        print(f"Extrayendo los datos de {self.pdf_path}")
        datos = {}

        # Diccionario de expresiones regulares
        patrones = {
            'pct_obra_civil': (r'Obra Civil\s+(\d+)%', lambda m: int(m) / 100),
            'pct_turbogenerador': (r'Grupo\s+[Tt]urbogenerador\s+(\d+)%', lambda m: int(m) / 100),
            'pct_electrico': (r'Equipos?\s+El[eé]ctric.*?(\d+)\s*%', lambda m: int(m) / 100),
            'pct_ingenieria': (r'Ingenier[íi]a\s+y\s+Direcci[oó]n\s+de\s+Obra\s+(\d+)%', lambda m: int(m) / 100),
            'ratio_eur_kw_2006': (r'(1[.,]500)\s*€/kW', lambda m: 1500.0),
            'mant_base_eur_anio': (r'(225[.,]000)\s*€/año', lambda m: 225000.0)
        }

        with pdfplumber.open(self.pdf_path) as pdf:
            for pag, page in enumerate(pdf.pages, 1):
                texto = page.extract_text() or ""

                for clave, (regex, transformador) in patrones.items():
                    if clave not in datos:
                        m = re.search(regex, texto)
                        if m:
                            datos[clave] = transformador(m.group(1))
                            datos[f'{clave}_ref'] = f"IDAE (2006) p.{pag}"
                            print(f" Hecho: {clave}: {datos[clave]} (pág. {pag})")

        return self._aplicar_fallback_si_necesario(datos)

    def _aplicar_fallback_si_necesario(self, datos):
        """Si la extracción por regex falla, aplicamos los valores por defecto ya conocidos."""
        esperados = ['pct_obra_civil', 'pct_turbogenerador', 'pct_electrico', 'pct_ingenieria', 'ratio_eur_kw_2006']
        faltantes = [c for c in esperados if c not in datos]

        if faltantes:
            print(f"Fallback activado para: {faltantes}")
            fallback = {
                'pct_obra_civil': 0.40, 'pct_turbogenerador': 0.30,
                'pct_electrico': 0.22, 'pct_ingenieria': 0.08, 'ratio_eur_kw_2006': 1500.0
            }
            for k in faltantes:
                datos[k] = fallback[k]
                datos[f'{k}_ref'] = "IDAE (2006) p.68-69 [Fallback manual]"
        return datos

In [4]:
class INEIPCClient:
    """Obtiene el factor de inflación oficial mediante la API JSON del INE.

    Nota: Desde enero 2026 el INE usa base 2025. El ratio 2006→2026 requiere
    encadenar las series base 2021 y base 2025. El fallback (×1.505) ya incorpora
    ese enlace, obtenido de la herramienta oficial varipc del INE.
    Ref: INE (2026). Herramienta VarIPC. https://www.ine.es/varipc/
    """

    # Series a intentar en orden. Si el INE actualiza el código de la
    _SERIES_CANDIDATAS = [
        'IPC251856',
    ]

    # Fallback oficial: INE varipc
    _FALLBACK = {
        'factor': 1.505,
        'ref':    ('INE (2026). IPC General. Herramienta VarIPC. '
                   'Abril 2006 → Abril 2026. https://www.ine.es/varipc/ '
                   '[Consultado 19/05/2026]'),
        'origen': 'Fallback documentado (base 2021→2025 encadenada)',
    }

    _HEADERS = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer':    'https://www.ine.es/',
        'Accept':     'application/json',
    }

    def obtener_factor(self, anio_base=2006, mes_base=4,
                       anio_fin=2026, mes_fin=4) -> dict:

        clave_base = f'{anio_base}M{mes_base:02d}'
        clave_fin  = f'{anio_fin}M{mes_fin:02d}'

        for codigo in self._SERIES_CANDIDATAS:
            try:
                factor, ref = self._intentar_serie(codigo, clave_base, clave_fin)
                print(f'¡ÉXITO! API INE ({codigo}). Factor: {factor:.4f}')
                return {'factor': factor, 'ref': ref, 'origen': f'API INE ({codigo})'}
            except Exception as e:
                print(f'Serie {codigo} no disponible: {e}')

        # Ninguna serie funcionó
        print(f'Usando fallback documentado: ×{self._FALLBACK["factor"]}')
        print('AVISO: el cambio de base 2021→2025 (enero 2026) invalida la '
              'comparación directa por API. El fallback ya incorpora el enlace '
              'oficial del INE.')
        return self._FALLBACK

    def _intentar_serie(self, codigo: str, clave_base: str, clave_fin: str):
        url = f'https://servicios.ine.es/wstempus/js/ES/DATOS_SERIE/{codigo}?nult=500'
        req = urllib.request.Request(url, headers=self._HEADERS)

        with urllib.request.urlopen(req, timeout=15) as r:
            data = json.loads(r.read())

        lista = data.get('Data', [])
        if not lista:
            raise ValueError('Respuesta vacía de la API')

        indice = {
            f"{d['Anyo']}M{d['FK_Periodo']:02d}": float(d['Valor'])
            for d in lista
        }

        if clave_base not in indice or clave_fin not in indice:
            claves_muestra = list(indice.keys())[:4]
            raise ValueError(
                f'Periodos {clave_base} o {clave_fin} no encontrados. '
                f'Muestra disponible: {claves_muestra}'
            )

        factor = indice[clave_fin] / indice[clave_base]
        ref = (f'INE. IPC General. API Tempus3. Serie {codigo}. '
               f'{clave_base} → {clave_fin}. '
               f'Consultado {datetime.now().strftime("%d/%m/%Y")}. '
               f'https://servicios.ine.es/wstempus/js/')
        return factor, ref

In [5]:
# Logica del programa

class ActualizadorEconomico:
    """Se encarga de aplicar la inflación y recalcular costes."""

    def __init__(self, importe_base: float, importe_actual: float, anio_base: int, anio_actual: int, ref: str = None):
        self.factor_ipc = importe_actual / importe_base
        self.ref = f"INE ({anio_actual}). IPC {anio_base}→{anio_actual}. VarIPC."

        print(f"\nFactor de actualización: ×{self.factor_ipc:.4f} ({self.ref})")

    def procesar(self, modelo: ParametrosHidro) -> ParametrosHidro:
        db = modelo.datos_base
        da = modelo.datos_actualizados

        # Aplicar factor
        r_base = db['ratio_eur_kw_2006']
        r_actual = r_base * self.factor_ipc

        da['ratio_eur_kw_actualizado'] = r_actual
        da['eur_kw_obra_civil'] = r_actual * db['pct_obra_civil']
        da['eur_kw_turbogenerador'] = r_actual * db['pct_turbogenerador']
        da['eur_kw_electrico'] = r_actual * db['pct_electrico']
        da['eur_kw_ingenieria'] = r_actual * db['pct_ingenieria']

        mant_base_kw = db.get('mant_base_eur_anio', 225000) / 5000
        da['mant_eur_kw_anio'] = mant_base_kw * self.factor_ipc

        modelo.metadatos['factor_ipc'] = self.factor_ipc
        modelo.metadatos['ref_inflacion'] = self.ref
        return modelo


In [6]:
# Exportamoos

class CSVExporter:
    """Se encarga de transformar el modelo en CSV."""

    @staticmethod
    def exportar(modelo: ParametrosHidro, ruta_salida: str):
        registro = {**modelo.metadatos, **modelo.datos_base, **modelo.datos_actualizados}
        registro['fecha_generacion'] = datetime.now().strftime('%Y-%m-%d')

        df = pd.DataFrame([registro])
        df.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
        print(f"\nDatos exportados exitosamente a: {ruta_salida}")


In [7]:
# Orquestador principal

def generar_modelo_economico(
        pdf_path:   str = 'documentos_2.1.7_Minicentrales_hidroelectricas_125f6cd9.pdf',
        csv_salida: str = 'parametros_costes.csv',
        anio_base:  int = 2006, mes_base: int = 4,
        anio_fin:   int = 2026, mes_fin:  int = 4,
) -> ParametrosHidro:

    print('=' * 55)
    print('PIPELINE COSTES')
    print('=' * 55)

    # Extraemos los datos del PDF oficial del IDAE
    modelo = ParametrosHidro()
    modelo.datos_base = IDAEPDFExtractor(pdf_path).extraer_parametros()

    # Obtenemos factor IPC del INE
    resultado_ipc = INEIPCClient().obtener_factor(anio_base, mes_base, anio_fin, mes_fin)

    # Actualizamos los costes con el factor IPC
    modelo = ActualizadorEconomico(
        importe_base   = 1000.0,
        importe_actual = resultado_ipc['factor'] * 1000.0,
        anio_base      = anio_base,
        anio_actual    = anio_fin,
        ref            = resultado_ipc['ref'],

    ).procesar(modelo)

    # Exportamos CSV
    CSVExporter.exportar(modelo, csv_salida)

    # Resumen
    da = modelo.datos_actualizados
    print(f'\nTotal Coste Actualizado: {da["ratio_eur_kw_actualizado"]:.2f} €/kW')
    print(f'   ├─ Obra Civil   40%:  {da["eur_kw_obra_civil"]:.2f} €/kW')
    print(f'   ├─ Turbogen.    30%:  {da["eur_kw_turbogenerador"]:.2f} €/kW')
    print(f'   ├─ Eléctrico    22%:  {da["eur_kw_electrico"]:.2f} €/kW')
    print(f'   └─ Ingeniería    8%:  {da["eur_kw_ingenieria"]:.2f} €/kW')
    return modelo


# Ejecutamos todo
modelo = generar_modelo_economico(
    pdf_path   = '/content/drive/MyDrive/TFG - UAX/DATOS/documentos_2.1.7_Minicentrales_hidroelectricas_125f6cd9.pdf',
    csv_salida = '/content/drive/MyDrive/TFG - UAX/CSV/parametros_costes.csv',
)

PIPELINE COSTES
Extrayendo los datos de /content/drive/MyDrive/TFG - UAX/DATOS/documentos_2.1.7_Minicentrales_hidroelectricas_125f6cd9.pdf
 Hecho: pct_obra_civil: 0.4 (pág. 70)
 Hecho: pct_turbogenerador: 0.3 (pág. 70)
 Hecho: pct_electrico: 0.22 (pág. 70)
 Hecho: pct_ingenieria: 0.08 (pág. 70)
 Hecho: ratio_eur_kw_2006: 1500.0 (pág. 71)
 Hecho: mant_base_eur_anio: 225000.0 (pág. 71)
Serie IPC251856 no disponible: Periodos 2006M04 o 2026M04 no encontrados. Muestra disponible: ['1984M05', '1984M06', '1984M07', '1984M08']
Usando fallback documentado: ×1.505
AVISO: el cambio de base 2021→2025 (enero 2026) invalida la comparación directa por API. El fallback ya incorpora el enlace oficial del INE.

Factor de actualización: ×1.5050 (INE (2026). IPC 2006→2026. VarIPC.)

Datos exportados exitosamente a: /content/drive/MyDrive/TFG - UAX/CSV/parametros_costes.csv

Total Coste Actualizado: 2257.50 €/kW
   ├─ Obra Civil   40%:  903.00 €/kW
   ├─ Turbogen.    30%:  677.25 €/kW
   ├─ Eléctrico    2